# Person search: Backbone + FPN → RPN → RoI Align → ReID

This is the main training notebook. The synthetic dataset exercises the pipeline;
it does not measure person-search accuracy.

1. ResNet (or the supported Swin backbone) produces C2–C5; FPN produces P2–P5.
2. RPN predicts objectness and box offsets for configurable tall anchors.
3. Per-level top-K selection, clipping, NMS and a final proposal budget produce detector RoIs.
4. Proposals are matched to person GT at IoU ≥ 0.5. GT supplies regression targets
   and identity labels, **never additional RoI boxes**.
5. Multi-scale RoI Align produces 7×7×256 features. Only positive proposals with
   known identity labels enter the ReID head and batch-hard triplet loss.

RPN uses binary cross entropy and positive-anchor Smooth L1. The existing auxiliary
RoI detection head also contributes classification and positive-proposal regression
losses, giving five loss terms in total. Proposal coordinates are detached; ReID
still backpropagates through pooled FPN features to the backbone.

P×K sampling improves identity coverage but cannot guarantee positive detector RoIs
or valid triplets. Log `_metric_valid_anchors`; zero metric loss is valid when no
triplets exist. DINO remains an unimplemented backbone adapter.


## Setup

Assumes `backbone.py`, `fpn.py`, `rpn.py`, `roi_heads.py`, `metric_loss.py`,
`model.py` are in the same directory as this notebook (or on `sys.path`).


In [ ]:
import sys, os, time, math, random
from pathlib import Path
src_dir = Path.cwd() if (Path.cwd() / "model.py").exists() else Path.cwd() / "src"
sys.path.insert(0, str(src_dir.resolve()))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
import matplotlib.pyplot as plt

from model import PersonSearchModel
from training import start_run, train_one_epoch, validate
import wandb
import gc

# In VS Code, select the project .venv/bin/python notebook kernel.
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB")

torch.manual_seed(0)
random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


## 1. Dataset

### Why synthetic, and what a real dataset needs to provide

Every image needs: a tensor `(3, H, W)`, a set of person GT boxes
`(N, 4)` in `x1y1x2y2`, and a set of identity ids `(N,)` where `-1` means
"a real person, but this dataset didn't annotate who they are" (this is
completely normal — CUHK-SYSU/PRW both have plenty of unlabeled
bystanders in scene images; only a subset of boxes carry an ID).

Critically: **the same identity id must appear across multiple *different*
images** in the dataset for the P×K sampler and triplet loss to have
anything to learn from (see Section 2). The synthetic dataset below
deliberately builds this structure in: a fixed pool of identities, each
placed into several different synthetic images with a slightly different
random box each time — mimicking "the same person appears in multiple
scene images taken at different times/places", which is exactly the real
structure of person-search datasets.


In [ ]:
class SyntheticPersonSearchDataset(Dataset):
    """
    Generates random images with a variable number of person boxes each.
    A shared pool of `num_identities` ids is reused across many images
    (each id shows up in `images_per_identity` different images), plus
    each image gets a few extra "unlabeled bystander" boxes (id=-1) to
    exercise the unlabeled-filtering logic in metric_loss.py.

    This is NOT meant to be visually meaningful (images are random noise)
    -- it exists purely to give every downstream shape/logic path (boxes,
    ids, batching, sampling) something structurally realistic to run on.
    """

    def __init__(self, num_identities=20, images_per_identity=4,
                 img_h=256, img_w=192, extra_unlabeled_per_image=(0, 2)):
        self.img_h, self.img_w = img_h, img_w
        self.extra_unlabeled_per_image = extra_unlabeled_per_image

        # build (image_index -> [(box, id), ...]) then flatten to a list of
        # "virtual images"; each identity's appearances are spread across
        # DIFFERENT virtual images so no single image contains duplicates
        # of the same identity (matches real datasets: one person, one box
        # per scene image).
        self.samples = []  # list of list[(box, id)]
        for _ in range(images_per_identity):
            # one "round": every identity appears exactly once this round,
            # each round assigned to a fresh image
            round_boxes = []
            for identity_id in range(num_identities):
                round_boxes.append([(self._random_person_box(), identity_id)])
            self.samples.extend(round_boxes)
        random.shuffle(self.samples)

        # merge some single-identity images into shared images with 2-3
        # identities each, to look more like real crowded scenes -- purely
        # cosmetic for this synthetic set, not required for correctness
        merged = []
        i = 0
        while i < len(self.samples):
            group_size = random.choice([1, 1, 2, 3])
            group = self.samples[i:i + group_size]
            merged.append([entry for img in group for entry in img])
            i += group_size
        self.samples = merged

        self.num_identities = num_identities

    def _random_person_box(self):
        # tall boxes, roughly person-shaped, placed randomly in the image
        w = random.uniform(25, 70)
        h = w * random.uniform(2.0, 3.2)
        x1 = random.uniform(0, max(1.0, self.img_w - w))
        y1 = random.uniform(0, max(1.0, self.img_h - h))
        return [x1, y1, x1 + w, y1 + h]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        entries = list(self.samples[idx])  # [(box, id), ...] labeled persons

        n_extra = random.randint(*self.extra_unlabeled_per_image)
        for _ in range(n_extra):
            entries.append((self._random_person_box(), -1))

        image = torch.randn(3, self.img_h, self.img_w)  # placeholder pixels
        boxes = torch.tensor([e[0] for e in entries], dtype=torch.float32)
        ids = torch.tensor([e[1] for e in entries], dtype=torch.int64)
        return image, {"boxes": boxes, "ids": ids}

    def identity_to_indices(self):
        """id -> list of dataset indices containing that id. Used by the
        P x K sampler below."""
        mapping = {}
        for idx, entries in enumerate(self.samples):
            for _, identity_id in entries:
                mapping.setdefault(identity_id, []).append(idx)
        return mapping


from prw_dataset import PRWPersonSearchDataset, prw_collate_fn
data_root = src_dir.parent / 'data'
dataset = PRWPersonSearchDataset(data_root, split='train')
print(f"PRW train dataset size: {len(dataset)} images")
img, tgt = dataset[0]
print(f"sample 0: image {tuple(img.shape)}, boxes {tuple(tgt['boxes'].shape)}, ids {tgt['ids'].tolist()}")


## 2. P×K identity-balanced batch sampler

Choose P identities and K distinct images for each, then deduplicate shared scene
indices. A batch can therefore contain fewer than P×K images and additional scene
identities. Selected identities have K appearances, but detector recall and RoI
sampling determine whether they survive into the metric loss. Multiple proposals
for one person can also form positive pairs, even with plain image sampling.


In [ ]:
class PKIdentitySampler(Sampler):
    """
    Yields batches of image INDICES (not batches of tensors -- collate_fn
    below turns indices into an actual padded batch). Each yielded batch
    selects P identities x K images-per-identity,
    with no duplicate image index within a batch even if a busy scene image
    happens to contain multiple sampled identities.
    """

    def __init__(self, dataset, P: int = 4, K: int = 2):
        self.dataset = dataset
        self.P, self.K = P, K
        self.identity_to_indices = dataset.identity_to_indices()
        # drop identities that don't appear in >=K distinct images -- can't
        # satisfy K for them, and forcing it would just repeat one image
        self.eligible_identities = [i for i, idxs in self.identity_to_indices.items()
                                     if len(set(idxs)) >= K]
        assert len(self.eligible_identities) >= P, (
            f"only {len(self.eligible_identities)} identities have >={K} images each, "
            f"need >={P} for P={P}. Lower P/K or add more images_per_identity to the dataset."
        )

    def __iter__(self):
        identities = self.eligible_identities.copy()
        random.shuffle(identities)
        for i in range(0, len(identities) - self.P + 1, self.P):
            batch_identities = identities[i:i + self.P]
            batch_indices = []
            for identity_id in batch_identities:
                candidates = list(set(self.identity_to_indices[identity_id]))
                chosen = random.sample(candidates, self.K) if len(candidates) >= self.K \
                         else random.choices(candidates, k=self.K)
                batch_indices.extend(chosen)
            batch_indices = list(dict.fromkeys(batch_indices))
            random.shuffle(batch_indices)
            yield batch_indices

    def __len__(self):
        return len(self.eligible_identities) // self.P


def collate_fn(batch):
    """
    Pads every image in the batch to the max H/W in that batch (top-left
    aligned, no rescaling) and stacks into one tensor. Boxes need no
    adjustment since padding is only added to the bottom/right.
    Real datasets with varying image sizes need exactly this same
    letterbox-and-pad step -- kept generic rather than assuming the
    synthetic dataset's fixed size, since real data won't be fixed-size.
    """
    images, targets = zip(*batch)
    max_h = max(img.shape[1] for img in images)
    max_w = max(img.shape[2] for img in images)

    padded = torch.zeros(len(images), 3, max_h, max_w)
    for i, img in enumerate(images):
        padded[i, :, :img.shape[1], :img.shape[2]] = img

    return padded, list(targets)


P, K = 2, 2  # at most four 256×192 scenes per batch for the 6 GB GPU
sampler = PKIdentitySampler(dataset, P=P, K=K)
loader = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_fn)
print(f"P={P}, K={K} -> up to {P*K} images per batch, {len(sampler)} batches/epoch")

images, targets = next(iter(loader))
print(f"batch images: {tuple(images.shape)}")
all_ids_this_batch = torch.cat([t['ids'] for t in targets])
labeled_ids = all_ids_this_batch[all_ids_this_batch >= 0]
from collections import Counter
print(f"labeled identity counts in this batch: {dict(Counter(labeled_ids.tolist()))}")
print("(selected identities have K image appearances; incidental identities may appear once)")


## 3. Training loop

Runs a handful of steps on the synthetic dataset. All 5 loss terms from
`model.py` are logged separately (not just their sum) specifically so we
can *watch* `metric_loss` / `_metric_valid_anchors` behave differently
depending on whether the P×K sampler is actually being used -- see the
comparison at the end of this section.


In [ ]:
# The reusable loop logs all five losses, their sum, proposal recall/counts,
# positive/labeled RoIs, valid triplets, LR, gradient norm, AMP scale,
# skipped updates, step time, throughput and peak GPU allocated/reserved memory.
# It uses mixed precision on CUDA and rejects non-finite losses/gradients.
from training import train_one_epoch, validate

# Use the held-out PRW split for validation.
val_dataset = PRWPersonSearchDataset(data_root, split='test')
val_loader = DataLoader(val_dataset, batch_size=2, collate_fn=collate_fn)


In [ ]:
# torchvision ratios are height / width; scale is sqrt(anchor area).
model_config = dict(
    backbone_name="resnet50", pretrained=False,
    anchor_sizes=((16,), (32,), (64,), (128,)),
    anchor_ratios=((2.0, 3.0, 4.0),) * 4,
    rpn_fg_iou_thresh=0.5, rpn_bg_iou_thresh=0.5,
    rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=512,
    rpn_pre_nms_top_n_test=500, rpn_post_nms_top_n_test=100,
)
run = start_run({**model_config, "P": P, "K": K, "data": "PRW",
                 "amp": device.type == "cuda", "lr": 1e-3}, name="prw-pk-check")
print(f"W&B mode: {'offline' if run.offline else 'online'}; directory: {run.dir}")
model = PersonSearchModel(**model_config).to(device)
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3, momentum=0.9, weight_decay=1e-4
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
history_pk = {}
for epoch in range(3):
    start_step = len(history_pk.get("total", []))
    epoch_history = train_one_epoch(model, loader, optimizer, device, max_steps=2,
                                   run=run, scaler=scaler, start_step=start_step)
    for key, values in epoch_history.items():
        history_pk.setdefault(key, []).extend(values)
    val_metrics = validate(model, val_loader, device, run=run,
                           step=len(history_pk["total"]) - 1, max_batches=2)
    print("validation:", val_metrics)


### Optional comparison with plain image sampling

Disabled by default to keep one model resident on the 6 GB GPU. If enabled, the
comparison gets its own W&B run after the main model and optimizer move to CPU.
Neither strategy guarantees valid triplets. Short runs on noise do not measure accuracy.


In [ ]:
RUN_PLAIN_COMPARISON = False
history_plain = None
if RUN_PLAIN_COMPARISON:
    model.cpu()
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value): state[key] = value.cpu()
    gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    run.finish()
    plain_run = start_run({**model_config, "data": "PRW", "sampler": "plain"},
                          name="prw-plain-check")
    plain_loader = DataLoader(dataset, batch_size=P*K, shuffle=True, collate_fn=collate_fn)
    model_plain = PersonSearchModel(**model_config).to(device)
    optimizer_plain = torch.optim.SGD([p for p in model_plain.parameters() if p.requires_grad],
                                      lr=1e-3, momentum=.9, weight_decay=1e-4)
    try:
        history_plain = train_one_epoch(model_plain, plain_loader, optimizer_plain,
                                         device, max_steps=6, run=plain_run)
    finally:
        plain_run.finish()
        del model_plain, optimizer_plain
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    model.to(device)


## 4. Loss curves

Not expecting convergence (6 steps, synthetic random-noise images, tiny
learning rate) -- just confirming every loss term is a real, finite,
backprop-connected number across steps, and comparing the two sampling
strategies' `metric_loss`/`valid_anchors` side by side.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
loss_keys = ["rpn_objectness_loss", "rpn_bbox_loss", "det_cls_loss",
             "det_bbox_loss", "metric_loss", "_metric_valid_anchors"]

for ax, key in zip(axes.flat, loss_keys):
    ax.plot(history_pk[key], marker="o", label="P x K sampler")
    if history_plain is not None:
        ax.plot(history_plain[key], marker="x", label="plain sampler")
    ax.set_title(key)
    ax.set_xlabel("step")
    ax.legend(fontsize=8)

plt.tight_layout()
artifact_dir = src_dir.parent / "artifacts"
artifact_dir.mkdir(exist_ok=True)
fig.savefig(artifact_dir / "loss_curves.png", dpi=140)
if not RUN_PLAIN_COMPARISON:
    run.log({"curves/losses": wandb.Image(fig)})
plt.show()


## 5. Checkpointing


In [ ]:
os.makedirs("checkpoints", exist_ok=True)
ckpt_path = "checkpoints/person_search_step6.pt"
torch.save({
    "model_config": model_config,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler.state_dict(),
    "step": len(history_pk["total"]),
}, ckpt_path)
print(f"saved checkpoint to {ckpt_path}")

# to resume:
# ckpt = torch.load(ckpt_path, map_location=device)
# model.load_state_dict(ckpt["model_state_dict"])
# optimizer.load_state_dict(ckpt["optimizer_state_dict"])


## Inference check: scene detections and query embeddings
The model is only smoke-trained; these outputs validate execution and alignment, not retrieval accuracy.


In [ ]:
model.eval()
images, _ = next(iter(val_loader))
with torch.inference_mode():
    results = model(images.to(device))
    query_embedding = model.embed_crop(images[:1, :, :128, :64].to(device))
assert torch.isfinite(query_embedding).all()
assert torch.allclose(query_embedding.norm(dim=1), torch.ones(1, device=device), atol=1e-4)
for result in results:
    assert all(torch.isfinite(v).all() for v in result.values())
    assert len(result["boxes"]) == len(result["scores"]) == len(result["embeddings"])
    similarities = query_embedding @ result["embeddings"].T
    print(f"detections: {len(result['boxes'])}, query similarities: {tuple(similarities.shape)}")
if not RUN_PLAIN_COMPARISON:
    run.log({"inference/detections": sum(len(r["boxes"]) for r in results),
             "inference/query_embedding_norm": float(query_embedding.norm())})
    run.finish()


## 6. Swapping in a real dataset (CUHK-SYSU / PRW)

This notebook's `SyntheticPersonSearchDataset` only needs to be replaced by
something that yields the same two things per item:

```python
image: torch.Tensor            # (3, H, W)
target: {
    "boxes": torch.Tensor,     # (N, 4), x1y1x2y2
    "ids":   torch.Tensor,     # (N,) int64, -1 for unlabeled persons
}
```

and additionally implements `identity_to_indices()` (id -> list of dataset
indices containing that id) so `PKIdentitySampler` keeps working unchanged.

**Left as a TODO rather than implemented here on purpose:** CUHK-SYSU and
PRW each ship their own annotation format (`.mat` files with dataset-
specific field layouts). Rather than guess at exact field names without
the actual files in hand to verify against, this notebook keeps the
dataset *interface* fixed and clearly specified above so you can write a
`CUHKSYSUDataset(Dataset)` / `PRWDataset(Dataset)` against the real
downloaded annotation files, cross-checking field names directly against
each dataset's `README`, and drop it into `loader = DataLoader(real_dataset,
batch_sampler=PKIdentitySampler(real_dataset, P=.., K=..), collate_fn=collate_fn)`
with nothing else in this notebook needing to change.

**Also not yet built:** `inference.py` (gallery embedding cache + query
matching + mAP/CMC evaluation) -- flagged back in the pipeline review as
the last missing piece; a natural next module once training on real data
is working.
